# Clientes com renegociação pendente: maiores valores

Análise exploratória para responder: **entre os clientes que fecharam uma negociação nos últimos dias, quais ainda estão com a renegociação pendente e têm os maiores saldos?**

**Fluxo**
1. Ler os logs de ligações (vários arquivos Parquet) com DuckDB.
2. Filtrar as negociações recentes das campanhas "extra".
3. Cruzar com o log de carga da carteira para saber a situação atual de cada conta.
4. Ficar com as contas em `RN_PENDENTE` e ordenar pelo saldo.

> ⚠️ **Os dados são fictícios** (gerados por `src/gerar_dados_sinteticos.py`). Rode o gerador antes de executar o notebook.


## 1. Leitura dos logs de ligações

In [1]:
import pandas as pd
from datetime import date
from pathlib import Path

pasta = Path("../data")

hoje = date.today()
mes_ano = hoje.strftime("%m-%Y")

In [2]:
import duckdb

con = duckdb.connect()

A view lê todos os arquivos Parquet do mês de uma vez. `union_by_name=True` junta arquivos mesmo que as colunas não sejam exatamente as mesmas.

In [3]:
con.execute(rf"""
create view lig as
SELECT *
FROM READ_PARQUET('{pasta.as_posix()}/*{mes_ano}.parquet', union_by_name=True)
""")

## 2. Ligações dos últimos 9 dias e campanhas existentes

In [4]:
con.execute("""

SELECT
    *,
    TRY_STRPTIME(DATA, '%d/%m/%Y')::DATE AS DATA_CONVERTIDA

FROM lig

WHERE TRY_STRPTIME(DATA, '%d/%m/%Y')::DATE
      BETWEEN CURRENT_DATE - INTERVAL '9 days'
      AND CURRENT_DATE""").df()

,DATA,HORA,AGENTE,CAMPANHA,LISTA,TELEFONE,CONEXAO,QUALIFICACAO,IDCONTATO_,tipo_ligacao,DATA_CONVERTIDA
0,08/10/2026,16:12:16,AGENTE_13,COB_EXTRA_2,EXTRA_2,00953108321,Não atendida,Sem contato,54613481,NaN,2026-10-08
1,08/10/2026,10:50:28,AGENTE_03,COB_EXTRA_1,EXTRA_1,00999138964,Caixa postal,Caixa postal,40715769,NaN,2026-10-08
2,08/10/2026,10:36:06,AGENTE_26,COB_EXTRA_2,EXTRA_2,00927715512,Atendida,Negociação incluída agora,73134512,NaN,2026-10-08
3,08/10/2026,19:52:05,AGENTE_25,COB_61A180,61A180,00997498328,Caixa postal,Caixa postal,28381478,NaN,2026-10-08
4,08/10/2026,11:52:18,AGENTE_22,COB_EXTRA_1,EXTRA_1,00907828508,Caixa postal,Caixa postal,7923394,NaN,2026-10-08
...,...,...,...,...,...,...,...,...,...,...,...
23995,16/10/2026,16:54:42,AGENTE_18,COB_EXTRA_2,EXTRA_2,00905030879,Caixa postal,Caixa postal,51202449,OUTBOUND,2026-10-16
23996,16/10/2026,13:56:23,AGENTE_08,COB_EXTRA_1,EXTRA_1,00923105595,Atendida,Número errado,76265537,OUTBOUND,2026-10-16
23997,16/10/2026,09:18:57,AGENTE_27,COB_EXTRA_1,EXTRA_1,00967699532,Atendida,Negociação incluída agora,28141792,OUTBOUND,2026-10-16
23998,16/10/2026,14:41:00,AGENTE_13,COB_EXTRA_3,EXTRA_3,00989100623,Atendida,Negociação incluída agora,29223565,OUTBOUND,2026-10-16


In [5]:
con.execute("""

SELECT
    DISTINCT CAMPANHA 
FROM lig
""").df()

,CAMPANHA
0,COB_31A60
1,COB_61A180
2,COB_EXTRA_3
3,COB_EXTRA_1
4,COB_PREVENTIVO
5,COB_EXTRA_2
6,COB_14A30
7,COB_RECEPTIVO


## 3. Negociações recentes nas campanhas extra

In [6]:
df_negociacoes = con.execute("""
    SELECT
        *
    FROM lig
    WHERE TRY_STRPTIME(DATA, '%d/%m/%Y')::DATE
          BETWEEN CURRENT_DATE - INTERVAL '9 days' AND CURRENT_DATE
      AND TRIM(QUALIFICACAO) ILIKE 'Negociação incluída agora'
      AND CAMPANHA IN (
          'COB_EXTRA_1',
          'COB_EXTRA_2'
      )
    ORDER BY TRY_STRPTIME(DATA, '%d/%m/%Y')::DATE DESC
""").df()

display(df_negociacoes)

print("Total de registros:", len(df_negociacoes))

,DATA,HORA,AGENTE,CAMPANHA,LISTA,TELEFONE,CONEXAO,QUALIFICACAO,IDCONTATO_,tipo_ligacao
0,16/10/2026,09:18:57,AGENTE_27,COB_EXTRA_1,EXTRA_1,00967699532,Atendida,Negociação incluída agora,28141792,OUTBOUND
1,16/10/2026,13:37:40,AGENTE_25,COB_EXTRA_2,EXTRA_2,00986603543,Atendida,Negociação incluída agora,10068630,OUTBOUND
2,16/10/2026,08:00:19,AGENTE_08,COB_EXTRA_1,EXTRA_1,00937935556,Atendida,Negociação incluída agora,70447824,OUTBOUND
3,16/10/2026,18:56:38,AGENTE_28,COB_EXTRA_1,EXTRA_1,00925592918,Atendida,Negociação incluída agora,22703332,OUTBOUND
4,16/10/2026,18:17:36,AGENTE_19,COB_EXTRA_1,EXTRA_1,00974310501,Atendida,Negociação incluída agora,81123693,OUTBOUND
...,...,...,...,...,...,...,...,...,...,...
279,08/10/2026,13:53:06,AGENTE_27,COB_EXTRA_1,EXTRA_1,00968177120,Atendida,Negociação incluída agora,8643859,NaN
280,08/10/2026,08:26:59,AGENTE_14,COB_EXTRA_1,EXTRA_1,00982857974,Atendida,Negociação incluída agora,68605039,NaN
281,08/10/2026,16:15:02,AGENTE_02,COB_EXTRA_1,EXTRA_1,00903543858,Atendida,Negociação incluída agora,48084363,NaN
282,08/10/2026,10:10:06,AGENTE_16,COB_EXTRA_2,EXTRA_2,00926528234,Atendida,Negociação incluída agora,52589908,NaN


Total de registros: 284


In [7]:
df_negociacoes.columns.tolist()

['DATA',
 'HORA',
 'AGENTE',
 'CAMPANHA',
 'LISTA',
 'TELEFONE',
 'CONEXAO',
 'QUALIFICACAO',
 'IDCONTATO_',
 'tipo_ligacao']

In [8]:
df_negociacoes['IDCONTATO_']

0      28141792
1      10068630
2      70447824
3      22703332
4      81123693
         ...   
279     8643859
280    68605039
281    48084363
282    52589908
283    90000209
Name: IDCONTATO_, Length: 284, dtype: str

## 4. Cruzamento com o log de carga da carteira

In [9]:
import pandas as pd 
log = pd.read_csv(f"../data/LOG_CARGA_{hoje:%d%m%Y}.csv", sep=";")

log.head()

,CONTA,ASSESSORIA_COBRANCA,DIAS_ATRASO_REAL,FAIXA_ATRASO,SITUACAO,LOJA,PRODUTO_CLIENTE,SALDO_TOTAL
0,2587375,ASSESSORIA_A,232,FASE_04,DEVEDOR,361,CREDIARIO,"356,93"
1,76943670,INTERNA,535,FASE_05,DEVEDOR,139,CARTAO,"2617,00"
2,82272164,ASSESSORIA_B,120,FASE_03,DEVEDOR,111,CARTAO,"1200,30"
3,13747730,ASSESSORIA_A,552,FASE_05,RENEGOCIACAO,268,EMPRESTIMO,"90,10"
4,23085744,INTERNA,162,FASE_03,DEVEDOR,379,CREDIARIO,"86,22"


In [10]:
df_negociacoes[df_negociacoes['IDCONTATO_'].isin(log['CONTA'].astype(str))]

,DATA,HORA,AGENTE,CAMPANHA,LISTA,TELEFONE,CONEXAO,QUALIFICACAO,IDCONTATO_,tipo_ligacao
0,16/10/2026,09:18:57,AGENTE_27,COB_EXTRA_1,EXTRA_1,00967699532,Atendida,Negociação incluída agora,28141792,OUTBOUND
1,16/10/2026,13:37:40,AGENTE_25,COB_EXTRA_2,EXTRA_2,00986603543,Atendida,Negociação incluída agora,10068630,OUTBOUND
2,16/10/2026,08:00:19,AGENTE_08,COB_EXTRA_1,EXTRA_1,00937935556,Atendida,Negociação incluída agora,70447824,OUTBOUND
3,16/10/2026,18:56:38,AGENTE_28,COB_EXTRA_1,EXTRA_1,00925592918,Atendida,Negociação incluída agora,22703332,OUTBOUND
4,16/10/2026,18:17:36,AGENTE_19,COB_EXTRA_1,EXTRA_1,00974310501,Atendida,Negociação incluída agora,81123693,OUTBOUND
...,...,...,...,...,...,...,...,...,...,...
278,08/10/2026,08:26:58,AGENTE_11,COB_EXTRA_1,EXTRA_1,00999932396,Atendida,Negociação incluída agora,38387341,NaN
279,08/10/2026,13:53:06,AGENTE_27,COB_EXTRA_1,EXTRA_1,00968177120,Atendida,Negociação incluída agora,8643859,NaN
280,08/10/2026,08:26:59,AGENTE_14,COB_EXTRA_1,EXTRA_1,00982857974,Atendida,Negociação incluída agora,68605039,NaN
281,08/10/2026,16:15:02,AGENTE_02,COB_EXTRA_1,EXTRA_1,00903543858,Atendida,Negociação incluída agora,48084363,NaN


In [11]:
lc = log[log['CONTA'].astype(str).isin(df_negociacoes['IDCONTATO_'])]

In [12]:
lc.columns.tolist()

['CONTA',
 'ASSESSORIA_COBRANCA',
 'DIAS_ATRASO_REAL',
 'FAIXA_ATRASO',
 'SITUACAO',
 'LOJA',
 'PRODUTO_CLIENTE',
 'SALDO_TOTAL']

Situações possíveis das contas que negociaram:

In [13]:
lc['SITUACAO'].unique()

<ArrowStringArray>
['RN_PENDENTE', 'RENEGOCIACAO', 'DEVEDOR']
Length: 3, dtype: str

## 5. Renegociações pendentes

In [14]:
mask = (lc['SITUACAO'] == 'RN_PENDENTE')

pendente = lc[mask].copy()

In [15]:
pendente.columns.tolist()

['CONTA',
 'ASSESSORIA_COBRANCA',
 'DIAS_ATRASO_REAL',
 'FAIXA_ATRASO',
 'SITUACAO',
 'LOJA',
 'PRODUTO_CLIENTE',
 'SALDO_TOTAL']

O saldo vem como texto com vírgula decimal (`"1234,56"`); converto para número antes de ordenar.

In [16]:
pendente['SALDO_TOTAL'] = pendente['SALDO_TOTAL'].replace(',', '.', regex=True).astype(float)

In [17]:
pendente.dtypes

CONTA                    int64
ASSESSORIA_COBRANCA        str
DIAS_ATRASO_REAL         int64
FAIXA_ATRASO               str
SITUACAO                   str
LOJA                     int64
PRODUTO_CLIENTE            str
SALDO_TOTAL            float64
dtype: object

## 6. Resultado: clientes com maior saldo

In [18]:
df_negociacoes['IDCONTATO_'] = df_negociacoes['IDCONTATO_'].astype("Int64")

resultado = pendente.merge(
    df_negociacoes[["IDCONTATO_", "TELEFONE"]],
    left_on="CONTA",
    right_on="IDCONTATO_",
    how="left"
)

display(
    resultado[
        ["CONTA", "TELEFONE", "SALDO_TOTAL", "DIAS_ATRASO_REAL"]
    ].sort_values(by="SALDO_TOTAL", ascending=False)
)

,CONTA,TELEFONE,SALDO_TOTAL,DIAS_ATRASO_REAL
37,66272859,00904974235,6952.20,80
28,49871096,00937408320,5239.51,71
85,64274899,00998356843,4765.69,283
41,74326827,00954540377,3329.49,308
133,60147110,00950973363,3013.33,730
...,...,...,...,...
119,65241545,00906738144,72.27,129
136,18137077,00993883694,68.90,223
101,83827984,00978402490,57.51,97
102,83827984,00978402490,57.51,97
